<a href="https://colab.research.google.com/github/jaalvalcan/GEE_index_sets/blob/main/dw_prithvi_copernicus_fm(s1).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import os

# 1. Force install compatible versions
# We use numpy 1.26.4 as the 'bridge' version
!pip install -q "numpy<2.0.0" geemap terratorch huggingface_hub

# 2. Programmatically restart the runtime to load the new NumPy
print("\n... RESTARTING RUNTIME TO APPLY CHANGES ...")
os.kill(os.getpid(), 9)

In [ ]:
import ee, geemap, torch, numpy as np
import matplotlib.pyplot as plt
from terratorch.registry import BACKBONE_REGISTRY
from scipy.ndimage import zoom

ee.Authenticate()
ee.Initialize(project='ee-')
roi = ee.Geometry.Polygon([[-65.735836, -9.770472], [-65.735836, -9.726992],
                           [-65.695496, -9.726992], [-65.695496, -9.770472],
                           [-65.735836, -9.770472]])

In [ ]:
import torch
# We import the generic class and the factory that builds it
from terratorch.models.backbones.prithvi_vit import PrithviViT

print("Initializing NASA Prithvi Model (Final Attempt)...")

try:
    # We build the architecture for Prithvi-100M
    # (Base ViT: 12 layers, 12 heads, 768 embed dim)
    model = PrithviViT(
        pretrained="ibm-nasa-geospatial/Prithvi-EO-1.0-100M",
        in_chans=6,
        img_size=224,
        num_frames=1,
        patch_size=16,
        embed_dim=768,
        depth=12,
        num_heads=12,
        qkv_bias=True
    ).eval()

    print("Success: Prithvi-100M initialized.")

except Exception as e:
    print(f"Direct class build failed. Error: {e}")
    # Fallback to the most generic timm-based loader if terratorch is broken
    import timm
    model = timm.create_model('vit_base_patch16_224', in_chans=6, num_classes=0).eval()
    print("Success: Loaded generic ViT-Base (No NASA weights).")

# Quick test to confirm it's working
dummy = torch.randn(1, 6, 224, 224)
with torch.no_grad():
    out = model(dummy)
    print(f"Model is active. Feature shape: {out.shape}")

In [ ]:
# Quick test to confirm it's working
dummy = torch.randn(1, 6, 224, 224)
with torch.no_grad():
    out = model(dummy)

    # Prithvi returns a tuple (usually list of features from different layers)
    # We take the last one [-1] which contains the final structural encoding
    final_features = out[-1]
    print(f"Model is active. Feature shape: {final_features.shape}")

In [ ]:
# Updated Radar logic to force HH/HV consistency
def get_data_consistent(year):
    start, end = f'{year}-01-01', f'{year}-06-30'
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).filterDate(start, end).median().clip(roi)
    dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(roi).filterDate(start, end).select('label').mode().clip(roi)

    # Force HH/HV for both years to ensure mathematical consistency
    s1 = ee.ImageCollection("COPERNICUS/S1_GRD") \
        .filterBounds(roi) \
        .filterDate(start, end) \
        .filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HH')) \
        .median().clip(roi).select(['HH', 'HV'])

    return {'s2': s2, 'dw': dw, 's1': s1}

In [ ]:
def get_data(year):
    start, end = f'{year}-01-01', f'{year}-06-30'

    # 1. Sentinel-2 & Dynamic World (Same as before)
    s2 = ee.ImageCollection("COPERNICUS/S2_SR_HARMONIZED").filterBounds(roi).filterDate(start, end).median().clip(roi)
    dw = ee.ImageCollection("GOOGLE/DYNAMICWORLD/V1").filterBounds(roi).filterDate(start, end).select('label').mode().clip(roi)

    # 2. Sentinel-1 (Radar) - FIXED FOR HOMOGENEITY
    s1_col = ee.ImageCollection("COPERNICUS/S1_GRD") \
        .filterBounds(roi) \
        .filterDate(start, end) \
        .filter(ee.Filter.eq('instrumentMode', 'IW'))

    # Explicitly filter for VV/VH to avoid the "mixed bands" error
    s1_vv = s1_col.filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'VV'))

    # Check if VV exists in this time range
    count = s1_vv.size().getInfo()

    if count > 0:
        s1 = s1_vv.median().clip(roi).select(['VV', 'VH'])
        print(f"Radar {year}: Found {count} images with VV/VH. Using VV.")
    else:
        # Fallback to HH if no VV exists at all
        s1_hh = s1_col.filter(ee.Filter.listContains('transmitterReceiverPolarisation', 'HH'))
        s1 = s1_hh.median().clip(roi).select(['HH', 'HV'])
        print(f"Radar {year}: No VV found. Using HH/HV mode.")

    return {'s2': s2, 'dw': dw, 's1': s1}

# Re-run the fetching
print("1. Fetching Homogeneous Satellite Data...")
d21, d25 = get_data(2021), get_data(2025)

# Now proceed to Step C (Inference) and Step D (Visualization)

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import zoom
import torch

# --- STEP C: INFERENCE ---
def run_prithvi_inference(img_ee):
    # Select the 6 bands Prithvi expects: Blue, Green, Red, Narrow NIR, SWIR1, SWIR2
    arr = geemap.ee_to_numpy(img_ee.select(['B2', 'B3', 'B4', 'B8', 'B11', 'B12']), region=roi, scale=30)

    # Normalize (0-10000 -> 0-1) and Resize to 224x224
    t = torch.from_numpy(np.transpose(arr, (2, 0, 1))).float().unsqueeze(0) / 10000.0
    t_resized = torch.nn.functional.interpolate(t, size=(224, 224))

    with torch.no_grad():
        # Extract features from the tuple (taking the last hidden state)
        features_tuple = model(t_resized)
        features_tensor = features_tuple[-1]

        # Reshape the 196 sequence into 14x14 spatial patches
        # Note: If shape is [1, 196], we reshape to [1, 14, 14]
        return features_tensor.reshape(1, 14, 14)

print("Running Prithvi Structural Analysis...")
p21_feat = run_prithvi_inference(d21['s2'])
p25_feat = run_prithvi_inference(d25['s2'])

# 1. PRITHVI MAP: Calculate Magnitude of Change (Structural Deviation)
prithvi_diff = torch.abs(p25_feat - p21_feat).squeeze().cpu().numpy()
prithvi_map = zoom(prithvi_diff, 16) # Upscale 14x14 to 224x224

# 2. DYNAMIC WORLD MAP: Categorical Loss (Trees to non-trees)
dw21_arr = geemap.ee_to_numpy(d21['dw'], region=roi, scale=30).squeeze()
dw25_arr = geemap.ee_to_numpy(d25['dw'], region=roi, scale=30).squeeze()
dw_loss = ((dw21_arr == 1) & (dw25_arr != 1)).astype(float)
dw_map = zoom(dw_loss, 224/dw_loss.shape[0])

# 3. RADAR MAP: Physical Change (Backscatter Deviation)
s1_21 = geemap.ee_to_numpy(d21['s1'], region=roi, scale=30)
s1_25 = geemap.ee_to_numpy(d25['s1'], region=roi, scale=30)
radar_diff = np.abs(s1_25 - s1_21).mean(-1)
radar_map = zoom(radar_diff, 224/radar_diff.shape[0])

# --- STEP D: VISUALIZATION ---
fig, ax = plt.subplots(1, 3, figsize=(22, 7))
titles = ["A. Dynamic World\n(Land Cover Class Shift)",
          "B. NASA Prithvi\n(Texture & Canopy Shift)",
          "C. Radar/Copernicus\n(Physical Biomass Shift)"]
maps = [dw_map, prithvi_map, radar_map]
cmaps = ['Greens', 'magma', 'viridis']

for i in range(3):
    im = ax[i].imshow(maps[i], cmap=cmaps[i])
    ax[i].set_title(titles[i], fontsize=15, pad=20)
    plt.colorbar(im, ax=ax[i], fraction=0.046, pad=0.04)
    ax[i].axis('off')

plt.tight_layout()
plt.show()

In [ ]:
from skimage.filters import threshold_otsu
from skimage.transform import resize
import numpy as np
import matplotlib.pyplot as plt

def process_and_resize(m, target_shape=(224, 224), is_binary=False):
    # 1. Clean NaNs
    m_clean = np.nan_to_num(m)
    # 2. Resize to the exact target shape
    # order=0 is 'Nearest Neighbor' which preserves the original data values
    m_resized = resize(m_clean, target_shape, order=0, preserve_range=True)

    if is_binary:
        return (m_resized > 0.5).astype(int)
    else:
        # For Prithvi and Radar, we calculate the threshold AFTER resizing
        if np.max(m_resized) == 0:
            return np.zeros_like(m_resized)
        thresh = threshold_otsu(m_resized)
        return (m_resized > thresh).astype(int)

print("Synchronizing all sensors to 224x224 grid...")

# We force EVERYTHING to (224, 224)
TARGET = (224, 224)

# 1. Prepare individual binary masks with forced resizing
bw_dw = process_and_resize(dw_map, TARGET, is_binary=True)
bw_prithvi = process_and_resize(prithvi_map, TARGET, is_binary=False)
bw_radar = process_and_resize(radar_map, TARGET, is_binary=False)

# 2. Generate the Weighted Result Map
# Now they are guaranteed to be the same shape
result_map = bw_dw + bw_prithvi + bw_radar

# 3. Visualization
plt.figure(figsize=(22, 7))
cmap_final = plt.cm.get_cmap('YlOrRd', 4)
im = plt.imshow(result_map, cmap=cmap_final)

cbar = plt.colorbar(im, ticks=[0, 1, 2, 3], fraction=0.046, pad=0.04)
cbar.ax.set_yticklabels(['Stable Forest', 'Subtle Disturbance', 'Probable Change', 'Confirmed Deforestation'])

plt.title("FINAL BENCHMARK: Aligned Multi-Sensor Consensus\n(DW + NASA Prithvi + S1 Radar)", fontsize=15)
plt.axis('off')
plt.show()

# Final stats for your conclusion
print(f"--- FINAL STATS ---")
print(f"Confirmed Deforestation (Red): {np.sum(result_map == 3)} pixels")
print(f"Probable Change (Orange): {np.sum(result_map == 2)} pixels")

In [ ]:
import pandas as pd
import seaborn as sns
from skimage.transform import resize

# 1. Ensure all maps are exactly 224x224 before flattening
TARGET = (224, 224)

# Resize raw intensities so the correlation is based on continuous values (more accurate)
dw_res = resize(dw_map.astype(float), TARGET, order=0)
prithvi_res = resize(prithvi_map.astype(float), TARGET, order=0)
radar_res = resize(radar_map.astype(float), TARGET, order=0)

# 2. Flatten and create DataFrame
data = {
    'Dynamic World': dw_res.flatten(),
    'NASA Prithvi': prithvi_res.flatten(),
    'Copernicus S1': radar_res.flatten()
}

df = pd.DataFrame(data)

# 3. Calculate Correlation
corr_matrix = df.corr()

# 4. Plotting the Heatmap
plt.figure(figsize=(10, 8))
sns.heatmap(corr_matrix, annot=True, cmap='RdYlGn', vmin=0, vmax=1, center=0.5)
plt.title("Scientific Benchmark: Cross-Model Correlation Matrix", fontsize=15)
plt.show()

# Print the findings for your LinkedIn post
print("--- Correlation Insights ---")
for col in corr_matrix.columns:
    top_corr = corr_matrix[col].sort_values(ascending=False)
    print(f"{col} correlates highest with {top_corr.index[1]}: {top_corr.values[1]:.2f}")